# 02 - Data Cleaning: Credit Card Default

**Input:** `../data/default of credit card clients.xls` (raw)  
**Output:**
- `../data/cleaned.csv`: encoded and scaled, ready for modeling (notebook 04)
- `../data/cleaned_readable.csv`: same rows before encoding and scaling, with real NT$ amounts and text labels, used for EDA (notebook 03) and Power BI

Every decision below is also written in `../docs/cleaning_log.md`.

**Workflow:**
1. Load
2. Rename columns (`PAY_0` -> `PAY_1`, target)
3. Check duplicates
4. Fix undocumented `EDUCATION` and `MARRIAGE` codes
5. Fix undocumented `PAY_` codes
6. Handle negative bills
7. Create new features
8. Treat outliers with IQR
9. Add text labels and save the readable file
10. Encode categorical columns
11. Scale numeric columns
12. Save `cleaned.csv`

## Step 1: Load

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# header=1 -> real column names are on the 2nd row
df = pd.read_excel("../data/default of credit card clients.xls", header=1)

print("Starting shape:", df.shape)

Starting shape: (30000, 25)


## Step 2: Rename Columns

- `PAY_0` -> `PAY_1`, so the repayment columns match `BILL_AMT1` and `PAY_AMT1` (1 = September for all three).
- `default payment next month` -> `default`, a short name with no spaces.

In [2]:
df = df.rename(columns={"PAY_0": "PAY_1", "default payment next month": "default"})

df.columns.tolist()

['ID',
 'LIMIT_BAL',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_1',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default']

## Step 3: Check Duplicates

In notebook 01 we found 35 rows that are copies of another row when `ID` is ignored. Let us look at them.

In [3]:
# keep=False marks BOTH rows of each duplicate pair
dup_rows = df.loc[df.drop(columns=["ID"]).duplicated(keep=False)]
print("Rows involved:", len(dup_rows))

bill_cols = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

# how many of them have zero bills AND zero payments in all 6 months
print("With no bills and no payments at all:", (dup_rows[bill_cols + pay_amt_cols] == 0).all(axis=1).sum())

Rows involved: 70
With no bills and no payments at all: 60


**Decision: keep them.** Each row has its own customer `ID`, and most are customers who never used their card. Two such customers with the same age, limit and education look exactly alike, but they are still two real customers.

## Step 4: Fix Undocumented `EDUCATION` and `MARRIAGE` Codes

Documented codes: `EDUCATION` 1 = graduate school, 2 = university, 3 = high school, 4 = others.  
`MARRIAGE` 1 = married, 2 = single, 3 = others.

The unknown codes are few, and we cannot know what they mean, so we move them into the existing **"others"** group.

In [4]:
print("EDUCATION before:"); print(df["EDUCATION"].value_counts().sort_index())

# 0, 5, 6 -> 4 (others)
df.loc[df["EDUCATION"].isin([0, 5, 6]), "EDUCATION"] = 4

print("EDUCATION after:"); print(df["EDUCATION"].value_counts().sort_index())

EDUCATION before:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64
EDUCATION after:
EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64


In [5]:
print("MARRIAGE before:"); print(df["MARRIAGE"].value_counts().sort_index())

# 0 -> 3 (others)
df.loc[df["MARRIAGE"] == 0, "MARRIAGE"] = 3

print("MARRIAGE after:"); print(df["MARRIAGE"].value_counts().sort_index())

MARRIAGE before:
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64
MARRIAGE after:
MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64


## Step 5: Fix Undocumented `PAY_` Codes

In the `PAY_` columns, 1-9 = months late and -1 = paid on time. -2 and 0 are not documented, but neither is a delay (-2 = no credit used, 0 = minimum paid).

We set -2, -1 and 0 all to **0**, so every `PAY_` column simply means **"months late"** (0 = not late).

In [6]:
pay_cols = ["PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

# how many values will change in each column
print("Values below 0 per column:"); print((df[pay_cols] < 0).sum())

# clip(lower=0) turns every negative value into 0 and keeps the rest
df[pay_cols] = df[pay_cols].clip(lower=0)

df["PAY_1"].value_counts().sort_index()

Values below 0 per column:
PAY_1     8445
PAY_2     9832
PAY_3    10023
PAY_4    10035
PAY_5    10085
PAY_6    10635
dtype: int64


PAY_1
0    23182
1     3688
2     2667
3      322
4       76
5       26
6       11
7        9
8       19
Name: count, dtype: int64

## Step 6: Handle Negative Bills

A negative bill means the customer paid **more** than they owed, so the bank owes them money. For this project the important fact is that they owe **nothing**, so negative bills become 0. This also keeps the ratio features (Step 7) from going negative.

In [7]:
print("Negative values per bill column:"); print((df[bill_cols] < 0).sum())
print("Customers with any negative bill:", (df[bill_cols] < 0).any(axis=1).sum())

df[bill_cols] = df[bill_cols].clip(lower=0)

print("Negative values left:", (df[bill_cols] < 0).sum().sum())

Negative values per bill column:
BILL_AMT1    590
BILL_AMT2    669
BILL_AMT3    655
BILL_AMT4    675
BILL_AMT5    655
BILL_AMT6    688
dtype: int64
Customers with any negative bill: 1930
Negative values left: 0


## Step 7: Create New Features

We create the features **before** capping outliers, so the ratios use the real amounts. A capped bill divided by a capped limit would give a wrong ratio.

| New feature | How | Why |
|-------------|-----|-----|
| `credit_utilization` | average bill of 6 months / `LIMIT_BAL` | how much of the limit the customer uses (above 1 = over the limit) |
| `avg_payment_delay` | average of `PAY_1` ... `PAY_6` | average months late over the 6 months |
| `payment_to_bill_ratio` | total paid / total billed (6 months), between 0 and 1 | share of the bills that was paid. No bills = 1 (nothing owed), and above 1 is capped at 1 |
| `months_delayed` | number of months with `PAY_` > 0 | how often the customer was late |
| `max_delay` | largest value of `PAY_1` ... `PAY_6` | worst delay in the 6 months |

In [8]:
# 1. average bill / credit limit
df["credit_utilization"] = df[bill_cols].mean(axis=1) / df["LIMIT_BAL"]

# 2. average months late
df["avg_payment_delay"] = df[pay_cols].mean(axis=1)

# 3. total paid / total billed
total_paid = df[pay_amt_cols].sum(axis=1)
total_billed = df[bill_cols].sum(axis=1)
print("Customers with total bill = 0:", (total_billed == 0).sum())

df["payment_to_bill_ratio"] = total_paid / total_billed
df.loc[total_billed == 0, "payment_to_bill_ratio"] = 1            # nothing billed -> nothing to pay -> fully paid
print("Ratios above 1 (capped to 1):", (df["payment_to_bill_ratio"] > 1).sum())
df["payment_to_bill_ratio"] = df["payment_to_bill_ratio"].clip(upper=1)

# 4. number of late months
df["months_delayed"] = (df[pay_cols] > 0).sum(axis=1)

# 5. worst delay
df["max_delay"] = df[pay_cols].max(axis=1)

new_features = ["credit_utilization", "avg_payment_delay", "payment_to_bill_ratio", "months_delayed", "max_delay"]
df[new_features].describe().round(3)

Customers with total bill = 0: 966
Ratios above 1 (capped to 1): 2938


,credit_utilization,avg_payment_delay,payment_to_bill_ratio,months_delayed,max_delay
count,30000.000,30000.000,30000.000,30000.000,30000.000
mean,0.373,0.281,0.346,0.834,0.682
std,0.352,0.602,0.381,1.554,1.074
min,0.000,0.000,0.000,0.000,0.000
25%,0.030,0.000,0.043,0.000,0.000
50%,0.285,0.000,0.106,0.000,0.000
75%,0.688,0.333,0.714,1.000,2.000
max,5.364,6.000,1.000,6.000,8.000


## Step 8: Treat Outliers with IQR

IQR rule: a value is an outlier if it is above `Q3 + 1.5 * IQR` (or below `Q1 - 1.5 * IQR`).

- We **cap** outliers instead of deleting rows, so no customer is lost.
- We cap the **money columns**: `LIMIT_BAL`, `BILL_AMT1-6`, `PAY_AMT1-6`. All their lower limits are below 0, so only the upper side changes.
- We do **not** cap `AGE`. Ages 61-79 are real customers, not errors. The new ratio features are already limited in Step 7.

In [9]:
money_cols = ["LIMIT_BAL"] + bill_cols + pay_amt_cols

# money columns become float so the caps (e.g. 162389.375) can be stored
df[money_cols] = df[money_cols].astype(float)

for col in money_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    lower = Q1 - 1.5 * IQR

    n_capped = (df[col] > upper).sum() + (df[col] < lower).sum()
    df.loc[df[col] > upper, col] = upper
    df.loc[df[col] < lower, col] = lower
    print(col, "| upper:", round(upper, 2), "| capped rows:", n_capped)

LIMIT_BAL | upper: 525000.0 | capped rows: 167
BILL_AMT1 | upper: 162389.38 | capped rows: 2398
BILL_AMT2 | upper: 155538.5 | capped rows: 2395
BILL_AMT3 | upper: 146412.5 | capped rows: 2468
BILL_AMT4 | upper: 132774.88 | capped rows: 2620
BILL_AMT5 | upper: 122831.75 | capped rows: 2724
BILL_AMT6 | upper: 121111.62 | capped rows: 2688
PAY_AMT1 | upper: 11015.0 | capped rows: 2745
PAY_AMT2 | upper: 11250.5 | capped rows: 2714


PAY_AMT3 | upper: 10677.5 | capped rows: 2598
PAY_AMT4 | upper: 9589.12 | capped rows: 2994
PAY_AMT5 | upper: 9700.0 | capped rows: 2945
PAY_AMT6 | upper: 9823.38 | capped rows: 2958


In [10]:
# check the new maximums
df[money_cols].describe().loc[["min", "max"]]

,LIMIT_BAL,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
min,10000.0,0.000,0.0,0.0,0.000,0.00,0.000,0.0,0.0,0.0,0.000,0.0,0.000
max,525000.0,162389.375,155538.5,146412.5,132774.875,122831.75,121111.625,11015.0,11250.5,10677.5,9589.125,9700.0,9823.375


## Step 9: Add Text Labels and Save the Readable File

Power BI and the EDA charts are easier to read with words than with codes, so we add label columns.

In [11]:
df["SEX_label"] = df["SEX"].map({1: "Male", 2: "Female"})
df["EDUCATION_label"] = df["EDUCATION"].map({1: "Graduate school", 2: "University", 3: "High school", 4: "Others"})
df["MARRIAGE_label"] = df["MARRIAGE"].map({1: "Married", 2: "Single", 3: "Others"})

# final check: no missing values
print("Missing values:", df.isna().sum().sum())
print("Shape:", df.shape)

df.to_csv("../data/cleaned_readable.csv", index=False)
print("Saved ../data/cleaned_readable.csv")

Missing values: 0
Shape: (30000, 33)


Saved ../data/cleaned_readable.csv


## Step 10: Encode Categorical Columns

| Column | Method |
|--------|--------|
| `SEX` | **label encoding**: 1 = female, 0 = male (only 2 values) |
| `EDUCATION`, `MARRIAGE` | **one-hot encoding** with `drop_first=True`. The numbers are labels, not amounts, so "university = 2" must not be read as "twice graduate school" |

`ID` is kept (not scaled) so model predictions can be matched back to customers. It is not used as a feature.

In [12]:
model_df = df.copy()

# drop the label columns (only needed for the readable file)
model_df = model_df.drop(columns=["SEX_label", "EDUCATION_label", "MARRIAGE_label"])

# SEX: 2 = female -> 1, 1 = male -> 0
model_df["SEX"] = (model_df["SEX"] == 2).astype(int)
model_df = model_df.rename(columns={"SEX": "is_female"})

# words instead of numbers so the new column names are readable
model_df["EDUCATION"] = model_df["EDUCATION"].map({1: "graduate", 2: "university", 3: "high_school", 4: "others"})
model_df["MARRIAGE"] = model_df["MARRIAGE"].map({1: "married", 2: "single", 3: "others"})

model_df = pd.get_dummies(model_df, columns=["EDUCATION", "MARRIAGE"], drop_first=True, dtype=int)

print("Shape after encoding:", model_df.shape)
model_df.columns.tolist()

Shape after encoding: (30000, 33)


['ID',
 'LIMIT_BAL',
 'is_female',
 'AGE',
 'PAY_1',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default',
 'credit_utilization',
 'avg_payment_delay',
 'payment_to_bill_ratio',
 'months_delayed',
 'max_delay',
 'EDUCATION_high_school',
 'EDUCATION_others',
 'EDUCATION_university',
 'MARRIAGE_others',
 'MARRIAGE_single']

## Step 11: Scale Numeric Columns

`LIMIT_BAL` is in the hundreds of thousands, while `PAY_1` is 0-8. `StandardScaler` changes each numeric column to mean 0 and standard deviation 1, so logistic regression treats them fairly.

`ID`, the target and the 0/1 columns are **not** scaled.

In [13]:
scale_cols = ["LIMIT_BAL", "AGE"] + pay_cols + bill_cols + pay_amt_cols + new_features

scaler = StandardScaler()
model_df[scale_cols] = scaler.fit_transform(model_df[scale_cols])

# check: mean ~0, std ~1
model_df[scale_cols].describe().loc[["mean", "std"]].round(2)

,LIMIT_BAL,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,...,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,credit_utilization,avg_payment_delay,payment_to_bill_ratio,months_delayed,max_delay
mean,0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,...,0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Step 12: Save `cleaned.csv`

In [14]:
print("Missing values:", model_df.isna().sum().sum())
print("Final shape:", model_df.shape)
print("Default rate (%):", round(model_df["default"].mean() * 100, 2))

model_df.to_csv("../data/cleaned.csv", index=False)
print("Saved ../data/cleaned.csv")

Missing values: 0
Final shape: (30000, 33)
Default rate (%): 22.12


Saved ../data/cleaned.csv


## Summary

- No rows removed: all **30,000** customers kept.
- Codes fixed: 345 `EDUCATION` rows -> 4 and 54 `MARRIAGE` rows -> 3. All negative `PAY_` codes -> 0.
- Negative bills -> 0. Money columns capped with IQR.
- 5 new features created.

See `../docs/cleaning_log.md` for every decision and its reason.